# Classify Local Images with SpeciesNet (eb_cameratrapai)

This notebook runs the **SpeciesNet** ensemble (detector + species classifier +
geofencing) on a local image folder, with an **optional MegaDetector v6 frontend**
that presorts images first.

## Deployment context

Field cameras in remote locations send images to a data center for classification.
To minimize data traffic, images are **presorted at the edge** with MegaDetector v6
so that blank frames (wind, rain, moving vegetation — often 70–95% of captures) are
dropped *before* transmission. Only frames containing an animal/person/vehicle are
sent on to SpeciesNet.

Step 3 reproduces that frontend so you can **test both configurations**:

- **With frontend** (`USE_MEGADETECTOR_FRONTEND = True`): MegaDetector v6 filters
  blanks; SpeciesNet classifies only the survivors. Reports how much data traffic
  the presort would save.
- **Without frontend** (`USE_MEGADETECTOR_FRONTEND = False`): every image goes
  straight to SpeciesNet (which runs its own detector internally).

Run [04_setup_speciesnet.ipynb](04_setup_speciesnet.ipynb) and
[01_setup_megadetector.ipynb](01_setup_megadetector.ipynb) first to install both
models. It is safe to run even when the image folder is empty.

## Step 1 — Define paths

In [ ]:
# find the machine we're running on, and set the repo/data/results roots accordingly. 
# This is used by the %%bash cells in the notebooks to set up the environment.

from pathlib import Path
import subprocess
import socket
import os

# Set MACHINE explicitly to force a specific MACHINE_PATHS row, or leave as
# None (optionally via the MACHINE env var) to auto-detect this box.
MACHINE = os.environ.get("MACHINE") or None

MACHINE_PATHS = {
    "spark": {
        "repo_root":    "/fs/work/git/ebio/project-id",
        "data_root":    "/fs/work/ebio/Data/Pictures",
        "results_root": "/fs/work/ebio/Results/project-id",
        "test_images":  "/fs/work/ebio/Data/pictures/fox_test",
    },
    "bigmac": {
        "repo_root":    "/Users/elhorte/git/ebio/project-id",
        "data_root":    "/Volumes/BigMacX/ebio/Pictures",
        "results_root": "/Volumes/BigMacX/ebio/project-id",
    },
    "bigmacx": {
        "repo_root":    "/Users/elhorte/git/ebio/project-id",
        "data_root":    "/Volumes/BigMacX/ebio/Data/Pictures",
        "results_root": "/Volumes/BigMacX/ebio/Results/project-id",
    },
    "MacBook": {
        "repo_root":    "/Users/elhorte/git/ebio/project-id",
        "data_root":    "/Users/elhorte/ebio/Data/Pictures",  
        "results_root": "/Users/elhorte/ebio/Results/project-id", 
    },
    "i9-14": {
        "repo_root":    "",   # TODO: set repo root for i9-14
        "data_root":    "",   # TODO: set data root for i9-14
        "results_root": "",   # TODO: set results root for i9-14
    },
}

def _detect_machine(table):
    """Pick the MACHINE_PATHS row for the box we're running on (see top note)."""
    override = os.environ.get("MACHINE_OVERRIDE")
    if override:
        if override not in table:
            raise ValueError(
                f"MACHINE_OVERRIDE={override!r} is not in MACHINE_PATHS "
                f"{sorted(table)}")
        return override, "MACHINE_OVERRIDE"
    # Filesystem truth: a row is viable only if BOTH its roots exist on this box.
    viable = [
        m for m, p in table.items()
        if p.get("repo_root") and os.path.isdir(p["repo_root"])
        and p.get("data_root") and os.path.isdir(p["data_root"])
    ]
    if len(viable) == 1:
        return viable[0], "filesystem"
    host = socket.gethostname().lower()
    # Hostname disambiguates when 0 or >1 rows' paths are visible.
    for m in (viable or list(table)):
        if m.lower() in host:
            return m, f"hostname({host})"
    if len(viable) > 1:
        return viable[0], f"filesystem(ambiguous:{viable})"
    raise RuntimeError(
        "Could not auto-detect MACHINE: no MACHINE_PATHS row has both its "
        f"repo_root and data_root present on host {host!r}, and no row name "
        "matches the hostname. Fill in MACHINE_PATHS or "
        f"`export MACHINE_OVERRIDE=<name>` (known: {sorted(table)})."
    )

if MACHINE is None:
    MACHINE, _machine_src = _detect_machine(MACHINE_PATHS)
else:
    if MACHINE not in MACHINE_PATHS:
        raise ValueError(
            f"MACHINE={MACHINE!r} is not a key of MACHINE_PATHS "
            f"(known: {sorted(MACHINE_PATHS)})"
        )
    _machine_src = "manual"

os.environ["MACHINE"] = MACHINE            # OUTPUT for the %%bash cells
_paths = MACHINE_PATHS[MACHINE]
_missing = [k for k, v in _paths.items() if not v]
if _missing:
    raise ValueError(
        f"MACHINE={MACHINE!r} is missing path(s) {_missing} in MACHINE_PATHS; "
        "fill them in before running.")

REPO         = _paths["repo_root"]
R            = _paths["data_root"]
RESULTS_ROOT = _paths["results_root"]
# Optional override for testing (defaults to this machine's data_root). Set via:
#   - the TEST_IMAGES env var, or
#   - a "test_images" entry in MACHINE_PATHS for this machine.
TEST_IMAGES = os.environ.get("TEST_IMAGES") or _paths.get("test_images")

repo_root = Path(REPO)
snet_root = repo_root / "third-party" / "eb_cameratrapai"
if TEST_IMAGES:
    image_dir = Path(TEST_IMAGES).expanduser()
else:
    image_dir = Path(R)                      # data_root for this machine
if not image_dir.is_dir():
    raise ValueError(
        f"Image directory does not exist: {image_dir!s} "
        f"(machine={MACHINE!r}, via {_machine_src}). "
        "Set the TEST_IMAGES env var to a valid folder, add a 'test_images' "
        "entry for this machine, or fix data_root in MACHINE_PATHS."
    )
os.environ["IMAGE_DIR"] = str(image_dir)     # OUTPUT for the %%bash cells
output_dir = Path(RESULTS_ROOT) / "speciesnet"
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Machine: {MACHINE} (via {_machine_src})")
print(f"Repository root: {repo_root}")
print(f"SpeciesNet path: {snet_root}")
print(f"Image directory: {image_dir}")
print(f"TEST_IMAGES override: {bool(TEST_IMAGES)}")
print(f"Output directory: {output_dir}")

## Step 2 — Check image folder contents

In [ ]:
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
images = sorted([p for p in image_dir.glob("**/*") if p.is_file() and p.suffix.lower() in image_exts])

print(f"Found {len(images)} image(s)")
for p in images[:20]:
    print(" -", p)

if not images:
    print("\nNo images found yet. Populate the folder and rerun this notebook.")

## Step 3 — (Optional) MegaDetector v6 presort frontend

Toggle `USE_MEGADETECTOR_FRONTEND` to test **with or without** the edge presort:

- `True` — run MegaDetector v6 on every image and keep only those with a detection
  at/above `FRONTEND_CONF_THRESHOLD` in `FRONTEND_KEEP_CATEGORIES`. This mirrors the
  field-camera blank-filter and reports the data-traffic reduction. The kept and
  filtered file lists are written as manifests in `output_dir`.
- `False` — pass every image straight to SpeciesNet.

Either way, the surviving images land in `selected_images`, which Step 5 classifies.

> Narrow `FRONTEND_KEEP_CATEGORIES` to `{"animal"}` for wildlife-only presorting, or
> keep `person`/`vehicle` to also forward human/vehicle activity.

In [ ]:
import sys
import subprocess
from pathlib import Path

# Editable installs register their paths via mechanisms Python only wires up at
# interpreter startup, so a kernel that predates the install can't import the
# packages until it restarts. Adding both checkouts to sys.path avoids that.
_repo = Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
for _p in (_repo / "third-party" / "eb_MegaDetector_v6" / "src",
           _repo / "third-party" / "eb_cameratrapai"):
    if _p.is_dir() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import json
import torch

# --- Frontend options ------------------------------------------------------
USE_MEGADETECTOR_FRONTEND = False                 # False -> send ALL images to SpeciesNet
FRONTEND_MODEL_VERSION    = "MDV6-yolov9-c"      # MegaDetector v6 variant for presort
FRONTEND_CONF_THRESHOLD   = 0.2                  # min detection confidence to keep an image
FRONTEND_KEEP_CATEGORIES  = {"animal", "person", "vehicle"}  # -> {"animal"} for wildlife-only
# ---------------------------------------------------------------------------

CLASS_NAMES = {0: "animal", 1: "person", 2: "vehicle"}
frontend_kept_txt = output_dir / "frontend_kept.txt"
frontend_filtered_txt = output_dir / "frontend_filtered.txt"

if not USE_MEGADETECTOR_FRONTEND:
    selected_images = list(images)
    n_filtered = 0
    print(f"Frontend DISABLED — all {len(selected_images)} image(s) go to SpeciesNet.")
else:
    from megadetector_ai import MegaDetectorV6

    if torch.cuda.is_available():
        _device = "cuda:0"
    elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        _device = "mps"
    else:
        _device = "cpu"

    print(f"Frontend ENABLED — loading MegaDetector v6 ({FRONTEND_MODEL_VERSION}) on {_device} ...")
    _md = MegaDetectorV6(device=_device, pretrained=True, version=FRONTEND_MODEL_VERSION)

    selected_images, filtered_images = [], []
    for i, img_path in enumerate(images):
        det = _md.single_image_detection(str(img_path))
        sv = det.get("detections")
        keep = False
        if sv is not None and len(sv.xyxy) > 0:
            for conf, cls_id in zip(sv.confidence, sv.class_id):
                if (float(conf) >= FRONTEND_CONF_THRESHOLD
                        and CLASS_NAMES.get(int(cls_id)) in FRONTEND_KEEP_CATEGORIES):
                    keep = True
                    break
        (selected_images if keep else filtered_images).append(img_path)
        if (i + 1) % 50 == 0 or (i + 1) == len(images):
            print(f"  Screened {i + 1}/{len(images)}")

    frontend_kept_txt.write_text("\n".join(str(p) for p in selected_images), encoding="utf-8")
    frontend_filtered_txt.write_text("\n".join(str(p) for p in filtered_images), encoding="utf-8")

    n_total, n_kept = len(images), len(selected_images)
    n_filtered = len(filtered_images)
    pct = (n_filtered / n_total * 100) if n_total else 0.0
    print(f"\nFrontend result: kept {n_kept}/{n_total}, filtered {n_filtered} "
          f"({pct:.1f}% would NOT be transmitted).")
    print(f"  Kept manifest:     {frontend_kept_txt}")
    print(f"  Filtered manifest: {frontend_filtered_txt}")

print(f"\n{len(selected_images)} image(s) will be classified by SpeciesNet.")

## Step 4 — Load the SpeciesNet model

Loads the full SpeciesNet ensemble (`DEFAULT_MODEL`). The model weights are
downloaded automatically on first use (~214 MB, no Kaggle login required) and
cached for later runs. SpeciesNet auto-detects a GPU if one is available.

> **First run downloads model weights**, so this cell can take a while before it
> finishes — that pause is the download, not a hang.

In [ ]:
import sys
import subprocess
from pathlib import Path

# Editable installs register their paths via mechanisms Python only wires up at
# interpreter startup, so a kernel that predates the install can't import the
# packages until it restarts. Adding both checkouts to sys.path avoids that.
_repo = Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
for _p in (_repo / "third-party" / "eb_MegaDetector_v6" / "src",
           _repo / "third-party" / "eb_cameratrapai"):
    if _p.is_dir() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from speciesnet import SpeciesNet, DEFAULT_MODEL, SUPPORTED_MODELS

print("Default SpeciesNet model:", DEFAULT_MODEL)
print("Supported (pytest-verified) models:", SUPPORTED_MODELS)

# Set geofence=False to disable country-based filtering of predictions.
model = SpeciesNet(DEFAULT_MODEL)
print("Model loaded.")

## Step 5 — Run SpeciesNet on the (presorted) images

Classifies `selected_images` — the frontend survivors from Step 3 (or all images
if the frontend is disabled) — and writes the raw SpeciesNet `predictions` JSON.
Each prediction includes the final `prediction` (a taxonomy string
`uuid;class;order;family;genus;species;common name`), a `prediction_score`, the
top-5 `classifications`, and `detections` (normalized `[x, y, width, height]`).

In [ ]:
import json

results_json = output_dir / "speciesnet_predictions.json"

if len(selected_images) == 0:
    raise RuntimeError(
        "No images to classify. Either the folder is empty or the MegaDetector "
        "frontend filtered everything out. Lower FRONTEND_CONF_THRESHOLD, widen "
        "FRONTEND_KEEP_CATEGORIES, or set USE_MEGADETECTOR_FRONTEND = False in Step 3."
    )

predictions_dict = model.predict(filepaths=[str(p) for p in selected_images])
results_json.write_text(json.dumps(predictions_dict, indent=2), encoding="utf-8")

preds = predictions_dict.get("predictions", [])
n_fail = sum(1 for p in preds if p.get("failures"))
print(f"Done. {len(preds)} prediction(s) written to: {results_json}")
if n_fail:
    print(f"{n_fail} image(s) had a component failure (see 'failures' in the JSON).")

## Step 6 — Preview predictions summary

In [ ]:
import json
from collections import Counter

results_json = output_dir / "speciesnet_predictions.json"
data = json.loads(results_json.read_text(encoding="utf-8"))
preds = data.get("predictions", [])


def common_name(prediction: str) -> str:
    """Last field of the 'uuid;class;...;common name' taxonomy string."""
    if not prediction:
        return "(none)"
    tail = prediction.split(";")[-1].strip()
    return tail or prediction


# Frontend effect (Step 3 variables are in scope after a sequential run).
if USE_MEGADETECTOR_FRONTEND:
    total_in = len(images)
    kept = len(selected_images)
    pct = ((total_in - kept) / total_in * 100) if total_in else 0.0
    print(f"Frontend: ON  — {kept}/{total_in} images forwarded, "
          f"{total_in - kept} filtered ({pct:.1f}% traffic saved)")
else:
    print(f"Frontend: OFF — all {len(images)} images classified")

print(f"Predictions: {len(preds)}")

label_counts = Counter(common_name(p.get("prediction", "")) for p in preds)
print("\nPredicted label distribution:")
for label, n in label_counts.most_common():
    print(f"  {n:4d}  {label}")

print("\nExamples:")
for p in preds[:10]:
    score = p.get("prediction_score")
    score_s = f"{score:.3f}" if isinstance(score, (int, float)) else "n/a"
    print(f"  {Path(p['filepath']).name}: {common_name(p.get('prediction',''))} ({score_s})")

## Step 7 — Visualize detections (bounding-box QA)

Draws SpeciesNet detection boxes on each classified image using the package's own
`draw_bboxes` helper, annotates the predicted species, and saves annotated copies
to an `annotated/` subfolder of the Step 1 `output_dir`.

Set `annotate_all = True` to annotate **every** image with detections; leave it
`False` to only process the first `max_preview` images. Up to `max_preview`
annotated images are shown inline so the notebook stays responsive.

In [ ]:
import sys
import subprocess
from pathlib import Path

# Editable installs register their paths via mechanisms Python only wires up at
# interpreter startup, so a kernel that predates the install can't import the
# packages until it restarts. Adding both checkouts to sys.path avoids that.
_repo = Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
for _p in (_repo / "third-party" / "eb_MegaDetector_v6" / "src",
           _repo / "third-party" / "eb_cameratrapai"):
    if _p.is_dir() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import json
from pathlib import Path

from PIL import ImageDraw, ImageFont
from IPython.display import Image as IPyImage, display
from speciesnet import draw_bboxes, load_rgb_image

results_json = output_dir / "speciesnet_predictions.json"
data = json.loads(results_json.read_text(encoding="utf-8"))
preds = data.get("predictions", [])

# --- Options ---------------------------------------------------------------
annotate_all = True             # True: annotate every image with detections
max_preview = 6                 # how many annotated images to show inline
clear_annotated_output = True   # clear prior *_annotated.png before writing new ones
# ---------------------------------------------------------------------------


def common_name(prediction: str) -> str:
    if not prediction:
        return "(none)"
    return prediction.split(";")[-1].strip() or prediction


annotated_dir = output_dir / "annotated"
annotated_dir.mkdir(parents=True, exist_ok=True)
if clear_annotated_output:
    for old in annotated_dir.glob("*_annotated.png"):
        old.unlink()
    print(f"Cleared existing annotated files in: {annotated_dir}")

with_dets = [p for p in preds if p.get("detections")]
print(f"{len(with_dets)} image(s) have detections")
if not with_dets:
    print("Nothing to visualize yet. Run Step 5 on a folder that contains animals/people/vehicles.")

to_process = with_dets if annotate_all else with_dets[:max_preview]
print(f"Annotating {len(to_process)} image(s) "
      f"({'all with detections' if annotate_all else 'preview only'}); "
      f"showing up to {max_preview} inline.")

saved = 0
for idx, p in enumerate(to_process):
    src = Path(p["filepath"])
    if not src.exists():
        print(f"  (skipped, missing file) {src}")
        continue

    # draw_bboxes returns a NEW composited image (it does not draw in place),
    # so we must capture the return value.
    img = load_rgb_image(str(src))
    img = draw_bboxes(img, p["detections"])

    # Caption the predicted species in the top-left corner.
    label = common_name(p.get("prediction", ""))
    score = p.get("prediction_score")
    caption = f"{label} {score:.2f}" if isinstance(score, (int, float)) else label
    draw = ImageDraw.Draw(img)
    draw.text((5, 5), caption, fill=(255, 255, 0), font=ImageFont.load_default())

    try:
        rel = src.relative_to(image_dir).with_suffix("")
        rel_key = "__".join(rel.parts)
    except ValueError:
        rel_key = src.stem
    safe_key = "".join(ch if (ch.isalnum() or ch in "-_.") else "_" for ch in rel_key)
    out_path = annotated_dir / f"{safe_key}_annotated.png"
    img.convert("RGB").save(out_path)
    saved += 1

    print(f"  [{idx + 1}/{len(to_process)}] {src.name}: {len(p['detections'])} box(es) "
          f"-> {label} -> {out_path.name}")
    if idx < max_preview:
        display(IPyImage(filename=str(out_path), width=700))

print(f"\nSaved {saved} annotated image(s) to: {annotated_dir}")